# 7.3 分布式 SpMV 通信数据流

## 本节学习目标

- 解释 Broadcast 和 AllGather
- 理解固定块和 padding

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 数据划分

行块大小为 `ceil(rows/world_size)`。rank r 负责自己的连续行区间；最后一 rank 不足 chunk 的输出以零填充，使所有 rank 的 AllGather send count 一致。

## 通信路径

rank 0 使用 HCCL Broadcast 同步完整 x；每个 rank 计算局部 y；HCCL AllGather 收集固定大小块；Host 侧按全局 rows 截掉 padding。

## 预期现象与结果分析

正确结果应与 CPU reference 一致。这里的局部 SpMV 调用 Ascend C RTC Kernel，Broadcast/AllGather 缓冲区和 collective 才在 ACL/HCCL Device 路径。

## 课后实践

以 rows=10、world=4 手工列出每个 rank 的区间、chunk 和 padding。

参考答案见 `answer/07.03_answer.md`。